# tutor

> The loop: pick, ask, let the learner struggle, grade, update, teach the smallest step.

This module runs the learning loop. `pick` chooses the concept and the depth, and a model writes the question.
The learner answers and can ask for hints. The model grades the answer, and the learner model updates the concept.
After a miss, the tutor teaches the smallest step. Each surface (web app, ramabana, notebook) uses `Tutor` and `Session`.

In [ ]:
#| default_exp tutor

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| export
import json, random, time, uuid
from fastcore.utils import *
from abhyasa.store import Learning
from abhyasa.model import DEPTHS, KIND_AT, GOAL, update, pick, recall, outcome
from abhyasa.llm import ask_json
from abhyasa.ground import VaultSource, ground, ground_text

## Tries and hints

A wrong answer does not end the question. The tutor names the first error, gives no fix, and the learner tries again.
After `MAX_TRIES` wrong tries, or when the learner stops, the tutor teaches the smallest step.
Then the state of the concept records the miss. A pass after a wrong try counts as `fragile`.

`HINTS` is the hint ladder, from the least to the most revealing. The model gets one rung for each hint request.

In [ ]:
#| export
MAX_TRIES = 3
HINTS = ('Point their attention at the part of the problem that matters. Name nothing.',
         'Describe the pattern or shape of the solution, in general terms.',
         'Name the category of idea or method that applies.',
         'Name the specific idea, rule or theorem to use, without using it.',
         'Outline the move: the first step and why, but not its result.',
         'Do one worked step of the solution. Leave the rest to them.')

`DEPTH_ASK` tells the model what a question at each depth must make the learner do.

In [ ]:
#| export
DEPTH_ASK = {1: 'recognise it: a multiple-choice question where each wrong option matches one specific misconception',
             2: 'recall it: state the rule, definition or fact in their own words',
             3: 'explain it: say why it is true or why it works, in their own words',
             4: 'apply it: solve a concrete problem that needs it; not a restatement',
             5: 'transfer it: use it in a new situation, or connect it to a different idea or subject'}

`TUTOR_SP` is the system prompt for questions, hints and lessons.

In [ ]:
#| export
TUTOR_SP = """You are a tutor whose only goal is that the learner gets better at thinking. You never think for them.
Rules:
- Never give the answer, or a hint that names the answer, before the learner tries.
- Ask questions that need reasoning, not recitation. Prefer new cases over the textbook example.
- Keep it short and plain. Use LaTeX between $...$ for maths. Use the learner's subject terms correctly.
- The learner is an adult expert-learner; never be patronising, never praise effort, never pad.
- Write in ASD-STE100 Simplified Technical English at a US grade 9 reading level: short sentences (at most 20 words),
  active voice, one instruction per sentence, simple words. Technical terms of the subject are allowed."""

## Prompts

Each prompt function makes the text for one model call. The prompt text is an instruction to the model.

`_concept_block` describes the concept and the learner. `question_prompt` asks for one question at one depth.

In [ ]:
#| export
def _concept_block(c, ctx=''):
    mis = '; '.join(c.get('misconceptions') or []) or 'none known'
    return (f"Concept: {c['name']} (subject: {c.get('subject') or 'general'})\nStatement: {c.get('statement') or '(none given)'}\n"
            f"Learner: floor {DEPTHS[c.get('floor', 0)]}, ceiling {DEPTHS[c['ceiling']] if c.get('ceiling', 6) < 6 else 'not found yet'}; "
            f"misconceptions seen: {mis}" + (f"\nFrom the learner's own vault (use only if relevant):\n{ctx}" if ctx else ''))

def question_prompt(c, depth, avoid=(), ctx=''):
    "The prompt that writes one question on `c` at `depth`."
    kind = KIND_AT[depth]
    shape = ('{"question": str, "options": [{"text": str, "misconception": str}], "correct": int, "answer": str, "rubric": [str]}'
             if kind == 'choice' else '{"question": str, "answer": str, "rubric": [str]}')
    av = '\n'.join(f'- {q}' for q in avoid)
    return (f"{_concept_block(c, ctx)}\n\nWrite ONE question that makes the learner {DEPTH_ASK[depth]}.\n"
            f"Aim it just past what they reliably know, so they must struggle a little. Ask ONE thing, in under 60 words. Write only the final question: no drafts, no corrections, no 'no: take'.\n"
            + (f"Target this misconception if it fits: {c['misconceptions'][-1]}\n" if c.get('misconceptions') else '')
            + (f"Do not repeat these earlier questions:\n{av}\n" if av else '')
            + f"`answer` is a model answer the learner will not see; `rubric` lists the 1-4 points a correct answer must have."
            + (" For options: 3 or 4, one correct (index in `correct`), each wrong one tied to a named misconception." if kind == 'choice' else '')
            + f"\nReturn JSON: {shape}")

`GRADE_SP` and `grade_prompt` grade one answer against the hidden model answer and the rubric.

In [ ]:
#| export
GRADE_SP = """You grade one answer. Judge the reasoning, including the think-aloud, not the wording.
- `correct`: true only when every rubric point is met in substance. A right result from wrong reasoning is not correct.
- `first_error`: where the reasoning first goes wrong, said in one sentence, WITHOUT the fix. Empty when correct.
- `misconception`: the wrong mental model behind the error, in under 12 words, or empty.
- `feedback`: when correct, confirm in one line and add one deeper link or consequence. When wrong, name the first error and give
  one nudge that makes them think; never give the answer.
- `grade`: 0-10.
- Write `feedback` and `first_error` in ASD-STE100 Simplified Technical English, grade 9: short sentences, active voice."""

def grade_prompt(q, answer, thinking='', confidence=50):
    "The prompt that grades one answer to question `q`."
    return (f"Concept: {q.name}\nDepth asked: {DEPTHS[q.depth]}\nQuestion: {q.question}\nModel answer (hidden from learner): {q.answer}\n"
            f"Rubric: {json.dumps(q.rubric)}\nEarlier wrong tries on this question: {json.dumps(q.tries)}\n\n"
            f"Learner's answer: {answer or '(none)'}\nLearner's think-aloud: {thinking or '(none)'}\nLearner's confidence: {confidence}/100\n\n"
            + (f"Passages from the learner's own sources (judge against these too, where they apply):\n{q.ctx}\n\n" if q.get('ctx') else '')
            + 'Return JSON: {"correct": bool, "grade": int, "first_error": str, "misconception": str, "feedback": str}')

`hint_prompt` asks for one rung of the hint ladder.

In [ ]:
#| export
def hint_prompt(q, level):
    "The prompt for hint `level` (0-5) on question `q`."
    given = '\n'.join(f'- {h}' for h in q.hints)
    return (f"Concept: {q.name}\nQuestion: {q.question}\nModel answer (hidden): {q.answer}\nWrong tries so far: {json.dumps(q.tries)}\n"
            + (f"Hints already given:\n{given}\n" if given else '')
            + f"\nGive hint level {level + 1} of 6: {HINTS[level]}\nOne or two sentences. Do not reveal the answer. Reply with the hint only.")

`teach_prompt` asks for the lesson after a miss. The lesson ends with a check question.

In [ ]:
#| export
def teach_prompt(q, first_error):
    "The prompt that teaches the smallest step after a miss."
    return (f"Concept: {q.name}\nStatement: {q.statement}\nQuestion: {q.question}\nModel answer: {q.answer}\n"
            f"Their tries: {json.dumps(q.tries)}\nTheir first error: {first_error or 'they did not know'}\n\n"
            + (f"Passages from their own sources; build on one and name it, if it fits:\n{q.ctx}\n\n" if q.get('ctx') else '')
            + "Teach the smallest step that fixes that first error, in under 120 words. Build on what they got right. "
            "Then show the model answer briefly. End with one line that starts 'Check:' and asks them to predict "
            "something new that the fixed idea implies (they will answer it next).")

## The tutor

`Tutor` keeps the open questions and the check questions from lessons.
`_ready_q` finds a question that came with the source of a concept.

In [ ]:
#| export
def _ready_q(c, depth, asked):
    "A ready question from the source of the concept at this depth, not asked before."
    qs = [q for q in c.get('questions') or () if q.get('depth', 2) == depth and q['q'] not in asked]
    return qs[0] if qs else None

#: functions `f(tutor, q, r)` that run after a question closes, in order; each can add to the result `r` (the demo module adds a demo)
CLOSE_HOOKS = []

class Tutor:
    "The learning loop over a `Learning` store with an `ask(prompt, sp)` model; `pool` runs background jobs."
    def __init__(self, learning:Learning, ask, goal:int=GOAL, context:bool=True, sources=None, pool=None):
        store_attr()
        if sources is None: self.sources = [VaultSource(learning)]
        self.open = {}     # question id -> question, while the learner works on it
        self.checks = {}   # concept -> (the check question of the lesson, the lesson), asked next on that concept
        self.jobs = {}     # concept -> a background job, for example a demo that a model makes

    def _ground(self, c) -> L:
        "Up to three passages on concept `c` from the sources of the learner, so a question can use what they read and did."
        if not (self.context and self.sources): return L()
        terms = [c['name'], *(c.get('aliases') or [])]
        try: return ground(' '.join(terms[:4]), self.sources, 3, must=terms)
        except Exception: return L()

## Ask a question

`ask_on` uses the first source that it finds: the check from the last lesson, a ready question, or a new question from the model.
It shuffles the options of a choice question. `public` removes the fields that the learner must not see.

In [ ]:
#| export
@patch
def ask_on(self:Tutor, c, depth:int, why:str='', session:str='') -> AttrDict:
    "Write a question on concept `c` at `depth`, keep it open, and return it without its answer fields."
    asked = self.learning.attempts(c['slug'], limit=20).attrgot('question')
    q = AttrDict(id=uuid.uuid4().hex[:10], concept=c['slug'], name=c['name'], statement=c.get('statement', ''),
                 subject=c.get('subject', ''), floor=c.get('floor', 0), depth=depth, kind=KIND_AT[depth], why=why, session=session,
                 hints=[], tries=[], at=time.time())
    ps = self._ground(c)
    q.update(ctx=ground_text(ps), grounds=[dict(src=p.src, ref=p.ref, title=p.title, url=p.url) for p in ps])
    if (ck := self.checks.pop(c['slug'], None)):
        q.update(question=ck[0], answer=ck[1], rubric=['the prediction follows from the lesson'], ready=True, why=f'{why}; the check from the lesson'.lstrip('; '))
    elif (r := _ready_q(c, depth, asked)): q.update(question=r['q'], answer=r.get('a', ''), rubric=r.get('rubric') or [r.get('a', '')], ready=True)
    else:
        r = ask_json(self.ask, question_prompt(c, depth, asked[:8], q.ctx), TUTOR_SP)
        q.update(question=r['question'], answer=r.get('answer', ''), rubric=r.get('rubric', []))
        if q.kind == 'choice' and r.get('options'):
            opts = list(enumerate(r['options']))
            random.shuffle(opts)
            q.options = [o for _, o in opts]
            q.correct = next(i for i, (j, _) in enumerate(opts) if j == r.get('correct', 0))
        elif q.kind == 'choice': q.kind = 'recall'
    self.open[q.id] = q
    return public(q)

def public(q) -> AttrDict:
    "A question as the learner can see it: no model answer, no rubric, no correct option, no misconception tags."
    out = AttrDict({k: v for k, v in q.items() if k not in ('answer', 'rubric', 'correct', 'options', 'statement', 'ctx', 'grounds')})
    if q.get('options'): out.options = [o['text'] for o in q.options]
    return out

`next` lets `pick` choose the concept, then asks a question on it.

In [ ]:
#| export
@patch
def next(self:Tutor, focus:str=None, subject:str=None, session:str='', skip=()) -> AttrDict|None:
    "The next question, chosen by `pick`, or None when there is nothing to learn or review."
    r = pick(self.learning.concepts(), focus=focus, subject=subject, goal=self.goal, skip=skip)
    return self.ask_on(*r, session=session) if r else None

## Hints

Each call to `hint` gives the next rung of the ladder. After the last rung, it gives the last hint again.

In [ ]:
#| export
@patch
def hint(self:Tutor, qid:str) -> str:
    "The next hint on the ladder, from the least to the most revealing."
    q = self.open[qid]
    if len(q.hints) >= len(HINTS): return q.hints[-1]
    h = self.ask(hint_prompt(q, len(q.hints)), TUTOR_SP).strip()
    q.hints.append(h)
    return h

## Grade an answer

`_grade` grades a choice question in code and other questions with the model.

In [ ]:
#| export
@patch
def _grade(self:Tutor, q, answer, thinking, confidence):
    if q.kind == 'choice' and q.get('options'):
        try: i = int(answer)
        except (TypeError, ValueError): i = next((j for j, o in enumerate(q.options) if o['text'] == answer), -1)
        ok = i == q.correct
        mis = '' if ok or not (0 <= i < len(q.options)) else q.options[i].get('misconception', '')
        return AttrDict(correct=ok, grade=10 if ok else 0, misconception=mis, first_error=mis and f'This option assumes: {mis}',
                        feedback='Correct.' if ok else 'Not this one. Look again at what each option assumes.')
    return AttrDict(ask_json(self.ask, grade_prompt(q, answer, thinking, confidence), GRADE_SP))

`answer` keeps the question open after a wrong try, with a nudge. A pass or the last try closes the question.

In [ ]:
#| export
@patch
def answer(self:Tutor, qid:str, answer:str='', thinking:str='', confidence:int=50, dont_know:bool=False) -> AttrDict:
    "Grade an answer. A wrong try keeps the question open with a nudge. A pass or the last try closes it and updates the concept."
    q = self.open[qid]
    if dont_know: return self.give_up(qid, confidence=confidence, dont_know=True)
    g = self._grade(q, answer, thinking, confidence)
    if not g.get('correct') and len(q.tries) + 1 < MAX_TRIES:
        q.tries.append(dict(answer=answer, thinking=thinking, confidence=confidence, first_error=g.get('first_error', ''),
                            misconception=g.get('misconception', '')))
        self.learning.record(q.concept, q.question, answer, thinking, False, confidence, q.depth, q.kind, len(q.hints),
                             grade=g.get('grade'), feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                             misconception=g.get('misconception', ''), session=q.session, final=False)
        return AttrDict(done=False, correct=False, feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                        tries_left=MAX_TRIES - len(q.tries))
    if not g.get('correct'):
        q.tries.append(dict(answer=answer, thinking=thinking, confidence=confidence, first_error=g.get('first_error', '')))
        return self._close(q, False, q.tries[0]['confidence'], g, teach=True)
    return self._close(q, True, confidence, g, answer=answer, thinking=thinking)

## Stop and teach

`give_up` closes the question as a miss. `_close` updates the concept, teaches after a miss and records the attempt.
Then each function in `CLOSE_HOOKS` can add to the result. The demo module uses this hook to attach a demo after a miss.
The confidence of the first try counts, because the first try shows what the learner believed.

In [ ]:
#| export
@patch
def give_up(self:Tutor, qid:str, confidence:int=0, dont_know:bool=False) -> AttrDict:
    "Close the question as a miss and teach the smallest step. A shown answer always counts as a miss."
    q = self.open[qid]
    first = q.tries[0] if q.tries else {}
    g = AttrDict(correct=False, grade=0, first_error=first.get('first_error', ''), misconception=first.get('misconception', ''),
                 feedback='')
    return self._close(q, False, first.get('confidence', confidence), g, teach=True, dont_know=dont_know and not q.tries)

In [ ]:
#| export
@patch
def _close(self:Tutor, q, correct, confidence, g, teach=False, answer='', thinking='', dont_know=False):
    c = self.learning.concept(q.concept)
    hints = len(q.hints) + 2 * bool(q.tries)   # a pass after a wrong try is never a clean pass
    u = update(c, q.depth, correct, confidence, hints, dont_know, g.get('misconception', ''))
    kind = u.pop('kind')
    self.learning.update(q.concept, **u)
    lesson = self.ask(teach_prompt(q, g.get('first_error', '')), TUTOR_SP).strip() if teach else ''
    if 'Check:' in lesson: self.checks[q.concept] = (lesson.rsplit('Check:', 1)[1].strip(), lesson)
    self.learning.record(q.concept, q.question, answer or (q.tries[-1]['answer'] if q.tries else ''), thinking, correct, confidence,
                         q.depth, q.kind, hints, dont_know, g.get('grade'), g.get('feedback', '') or lesson,
                         g.get('first_error', ''), g.get('misconception', ''), session=q.session, final=True, outcome=kind)
    del self.open[q.id]
    r = AttrDict(done=True, correct=correct, outcome=kind, feedback=g.get('feedback', ''), first_error=g.get('first_error', ''),
                 lesson=lesson, answer=q.answer if not correct else '', floor=u.floor, ceiling=u.ceiling,
                 due=u.due, depth=q.depth, grounds=q.get('grounds', []), concept=q.concept, session=q.session)
    for f in CLOSE_HOOKS: r = f(self, q, r) or r
    return r

## A session

`Session` keeps one sitting together. Its id goes on each attempt, and it stops after `limit` questions.
It asks about one concept for at most `run` questions in a row.
This lets the range between floor and ceiling close, and it keeps the learner interested.

In [ ]:
#| export
class Session:
    "One sitting: questions, answers and a summary filed at the end."
    def __init__(self, tutor:Tutor, focus:str=None, subject:str=None, limit:int=12, run:int=3):
        store_attr()
        self.id, self.start, self.log, self.q = uuid.uuid4().hex[:10], time.time(), [], None

    def _skip(self):
        last = [r['concept'] for r in self.log[-self.run:]]
        return {last[0]} if len(last) == self.run and len(set(last)) == 1 else set()

    def next(self) -> AttrDict|None:
        "The next question, or None when the session is over."
        if len(self.log) >= self.limit: return None
        skip = self._skip()
        self.q = self.tutor.next(focus=self.focus if self.focus not in skip else None, subject=self.subject, session=self.id, skip=skip)
        return self.q

    def answer(self, *args, **kw) -> AttrDict:
        r = self.tutor.answer(self.q.id, *args, **kw)
        if r.done: self.log.append(dict(concept=self.q.concept, depth=self.q.depth, correct=r.correct, outcome=r.outcome))
        return r

    def give_up(self, **kw) -> AttrDict:
        r = self.tutor.give_up(self.q.id, **kw)
        self.log.append(dict(concept=self.q.concept, depth=self.q.depth, correct=False, outcome=r.outcome))
        return r

    def hint(self) -> str: return self.tutor.hint(self.q.id)

    def summary(self) -> str:
        "A short summary, filed on the shelf as a `session` document."
        n, ok = len(self.log), sum(r['correct'] for r in self.log)
        cs = sorted({r['concept'] for r in self.log})
        s = f"{ok} of {n} passed. Concepts: {', '.join(cs) or 'none'}."
        if n: self.tutor.learning.log_session(self.id, s, start=self.start, end=time.time(), concepts=cs, passed=ok, asked=n)
        return s

## Tests

A fake model plays the tutor. Its grader passes an answer only when the answer has the words "rank position".
Two concepts are on the shelf, and `Reciprocal rank fusion` needs `Ranking`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake

lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', order=0)
lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml', prereqs=['ranking'], order=1)
def grader(p, sp): return dict(correct='rank position' in p.split("Learner's answer:")[1], grade=7, first_error='You add raw scores.',
                               misconception='scores are comparable across systems', feedback='Look at what each list really shares.')
fake = Fake({'You grade one answer': grader, 'Give hint level': 'Look at what the two lists have in common.',
             'Teach the smallest step': 'Only positions are shared. Check: what happens with k = 0?',
             'Write ONE question': dict(question='Why does RRF use ranks, not scores?', answer='Scores are not comparable.', rubric=['ranks are comparable'])})
t = Tutor(lr, fake, context=False)

The first question is on the prereq, at depth 3. The learner does not see the answer or the rubric.

In [ ]:
q = t.next()
test_eq((q.concept, q.depth, q.kind), ('ranking', 3, 'explain'))   # rrf waits for its prereq
assert 'answer' not in q and 'rubric' not in q

A wrong answer keeps the question open and names the first error. Then a hint comes from the ladder.

In [ ]:
r = t.answer(q.id, 'sum the scores', confidence=90)
test_eq((r.done, r.correct, r.tries_left), (False, False, 2))
test_eq(r.first_error, 'You add raw scores.')
test_eq(t.hint(q.id), 'Look at what the two lists have in common.')

A pass after a wrong try and a hint is `fragile`. The floor stays, and the concept is `shaky` at depth 3.

In [ ]:
r = t.answer(q.id, 'use the rank position', thinking='positions are shared', confidence=90)
test_eq((r.done, r.correct, r.outcome), (True, True, 'fragile'))   # passed, but after a wrong try and a hint
c = lr.concept('ranking')
test_eq((c.floor, c.shaky, c.seen), (0, 3, 1))
test_eq(len(lr.attempts('ranking')), 2)

Three wrong tries close the question, keep the misconception and teach.

In [ ]:
q = t.next(focus='ranking')
test_eq(q.depth, 3)    # shaky at explain: ask at explain again
for _ in range(3): r = t.answer(q.id, 'sum the scores', confidence=85)
test_eq((r.done, r.outcome), (True, 'misconception'))
assert 'Check:' in r.lesson
c = lr.concept('ranking')
test_eq((c.ceiling, c.misconceptions), (3, ['scores are comparable across systems']))

The check question from the lesson comes next.

In [ ]:
q = t.next(focus='ranking')
test_eq(q.question, 'what happens with k = 0?')   # the lesson's check comes next
t.open.clear()

A ready question from a source comes before a question from the model.

In [ ]:
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', questions=[dict(q='Join na + asti.', a='nāsti', depth=3)])
q = t.next(subject='sanskrit')
test_eq(q.question, 'Join na + asti.')

Code grades a choice question. A wrong option gives the misconception that it matches.

In [ ]:
fake.rules['Write ONE question'] = dict(question='Which is a sandhi?', options=[dict(text='na + asti → nāsti', misconception=''),
    dict(text='na + asti → naasti', misconception='vowels just sit side by side')], correct=0, answer='', rubric=[])
t.open.clear()
lr.update('sandhi', floor=0, ceiling=2)
q = t.next(subject='sanskrit')
test_eq(q.kind, 'choice')
r = t.answer(q.id, q.options.index('na + asti → naasti'), confidence=90)
test_eq(r.first_error, 'This option assumes: vowels just sit side by side')

A session stops after `limit` questions and files a summary.

In [ ]:
fake2 = Fake({k: fake.rules[k] for k in ('You grade one answer', 'Teach the smallest step')},
             dflt=dict(question='What does a rank tell you?', answer='Its position.', rubric=['position']))
s = Session(Tutor(temp_learning(), fake2, context=False), limit=2)
s.tutor.learning.put('Ranking', 'Order items by a score.', 'ml')
assert s.next() is not None
s.give_up()
assert s.next() is not None
test_eq(s.answer('the rank position', confidence=80).correct, True)
test_eq(s.next(), None)
test_eq(s.summary(), '1 of 2 passed. Concepts: ranking.')

The sources of the learner ground the question, the grade and the lesson. The learner sees the citations only when the question closes.

In [ ]:
from abhyasa.ground import passage

In [ ]:
class Notes:
    name = 'notes'
    def search(self, q, k=4): return L([passage('vault:notes', 'n1', 'My ranking notes', 'Ranking orders items; RRF adds 1/(k + rank).')])
gl = temp_learning(); gl.put('Ranking', 'Order items by a score.', 'ml')
gt = Tutor(gl, fake2, sources=[Notes()])
q = gt.next()
assert 'My ranking notes' in fake2.calls[-1][0]                 # the question prompt has the passage
assert 'grounds' not in q and 'ctx' not in q                  # the learner does not see it yet
r = gt.answer(q.id, 'the rank position', confidence=80)
test_eq(r.grounds, [dict(src='vault:notes', ref='n1', title='My ranking notes', url='')])
assert 'My ranking notes' in fake2.calls[-1][0]                 # the grade prompt has it too

A function in `CLOSE_HOOKS` sees each closed question and can add to the result. The result names its concept.

In [ ]:
seen = []
def spy(t, q, r): seen.append(q.concept); r.extra = 1
CLOSE_HOOKS.append(spy)
try: r = gt.answer(gt.next().id, 'the rank position', confidence=80)
finally: CLOSE_HOOKS.remove(spy)
test_eq((seen, r.extra, r.concept), (['ranking'], 1, 'ranking'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()